# **Project 'Wings Of Help'**

## **Python Code for Creating the Wings of Help Dataset**

The Wings of Help web service provides information interaction between individuals and organizations in need of assistance and volunteers by creating, publishing, viewing and processing requests for assistance, as well as tracking the status of their implementation.

The product provides a web solution within which:
  - Individuals or organizations can create requests for assistance.
  - Volunteers can view, select and execute current requests.
  - Administrators can moderate content and manage the platform.

The web service is developed as an MVP (Minimum Viable Product) with a basic set of functions, focused on quick launch and with the further possibility of scaling, expanding functionality or integrating with other services.

# **1. Creating dataset 'Wings Of Help'**

In [ ]:
"""
Wings Of Help — synthetic dataset generator
Creates CSV files:
- help_categories.csv
- users.csv
- help.csv
- events.csv (recommended for KPI calculations)

"""

from __future__ import annotations

import os
import random
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Optional, Tuple

import pandas as pd

## **1.1. Help category configuration**

In [ ]:
CATEGORIES = [
    "Evacuation / Relocation",
    "Medical Support",
    "Shelter / Housing",
    "Food & Basic Supplies",
    "Logistics / Transportation",
    "Psychological Support",
    "Child & Family Support",
    "Legal / Administrative Assistance",
    "Employment / Livelihoods",
    "Education / Tutoring",
    "Volunteer Coordination / Community Support",
    "Animal / Pet Assistance",
    "Other",
]

## **1.2. Determining the list of cities to provide assistance**

In [ ]:
CITIES = [
    "Kyiv", "Kharkiv", "Odesa", "Dnipro", "Zaporizhzhia", "Izium", "Marinka", "Pokrovsk",
    "Lviv", "Kryvyi Rih", "Mykolaiv", "Mariupol", "Vinnytsia", "Kherson", "Poltava",
    "Chernihiv", "Cherkasy", "Sumy", "Rivne", "Lutsk", "Ternopil", "Ivano-Frankivsk",
    "Uzhhorod", "Chernivtsi", "Zhytomyr", "Kropyvnytskyi", "Melitopol", "Berdyansk",
    "Bila Tserkva", "Kremenchuk", "Brovary", "Bakhmut"
]

## **1.3. Definition of the list of Ukrainian mobile operators**

In [ ]:
UA_MOBILE_PREFIXES = [
    # Kyivstar
    ("39", "Kyivstar"),
    ("67", "Kyivstar"),
    ("68", "Kyivstar"),
    ("77", "Kyivstar"),
    ("96", "Kyivstar"),
    ("97", "Kyivstar"),
    ("98", "Kyivstar"),

    # Vodafone Ukraine
    ("50", "Vodafone"),
    ("66", "Vodafone"),
    ("75", "Vodafone"),
    ("95", "Vodafone"),
    ("99", "Vodafone"),

    # lifecell
    ("63", "lifecell"),
    ("73", "lifecell"),
    ("93", "lifecell"),

    # Other / legacy
    ("89", "Intertelecom"),
    ("91", "Trimob"),
    ("92", "PEOPLEnet"),
    ("94", "Intertelecom"),
]

## **1.4. Helpers**

The Helpers block is a set of utility functions that do not directly create data but support and simplify the logic of generating a dataset.

They are needed to:
avoid code duplication in different parts of the generator;
make data realistic and consistent (events, emails, phones, roles);
Isolate small, technical logic from business logic (e.g., creating users, help, events).

For example:
  - utc_now() and rand_dt_between() are responsible for the correct generation of timestamps;
  - slugify() generates valid emails;
  - pick_weighted() allows you to generate roles and event types with the desired distribution;
  - generate_ua_phone() creates realistic Ukrainian mobile numbers.

In [ ]:
def utc_now() -> datetime:
    return datetime.now(timezone.utc)

def rand_dt_between(start: datetime, end: datetime, rng: random.Random) -> datetime:
    """Random datetime between start and end."""
    if end <= start:
        return start
    delta_seconds = int((end - start).total_seconds())
    return start + timedelta(seconds=rng.randint(0, delta_seconds))

def slugify(s: str) -> str:
    """Simple slug for emails; keeps ASCII letters/digits."""
    s = s.lower().replace("&", "and").replace("/", " ").replace("-", " ")
    keep = []
    for ch in s:
        if ch.isalnum():
            keep.append(ch)
        elif ch.isspace():
            keep.append("_")
    out = "".join(keep)
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")

def pick_weighted(items: List[Tuple[str, float]], rng: random.Random) -> str:
    """Pick item by weights."""
    total = sum(w for _, w in items)
    r = rng.random() * total
    upto = 0.0
    for val, w in items:
        upto += w
        if upto >= r:
            return val
    return items[-1][0]

def generate_ua_phone(rng: random.Random) -> str:
    """
    Generate realistic Ukrainian mobile phone number.
    Returns:
      phone_number: +380XXXXXXXXX
    """
    prefix, _ = rng.choice(UA_MOBILE_PREFIXES)
    rest = rng.randint(0, 9_999_999)
    phone_number = f"+380{prefix}{rest:07d}"
    return phone_number

## **1.5. Creating text templates for the title and description of requests/proposals**

To create the titles of requests and offers for assistance, as well as their detailed descriptions, several possible options for requests/offers in each assistance category, along with potential descriptions, are provided.

In [ ]:
TITLE_TEMPLATES: Dict[str, List[str]] = {
    "Food & Basic Supplies": [
        "Food & Hygiene Supplies Needed for IDP Family",
        "Urgent Food Packages Needed for Displaced Seniors",
        "Baby Formula and Hygiene Items Needed",
        "Basic Groceries Needed After Relocation",
    ],
    "Logistics / Transportation": [
        "Transportation Needed for Evacuation",
        "Ride Needed for Medical Appointment",
        "Delivery Assistance Needed for Supplies",
        "Transport Support Needed for Family Relocation",
    ],
    "Shelter / Housing": [
        "Temporary Housing Needed for Mother and Child",
        "Short-Term Accommodation Needed for Displaced Family",
        "Shelter Needed for Two Weeks",
        "Room Needed for Elderly Person",
        "Temporary Shelter for Elderly Woman",
    ],
    "Medical Support": [
        "Medication Support Needed for Chronic Condition",
        "Assistance Needed for Post-Surgery Care",
        "Help Needed to Access Medical Supplies",
        "Urgent Medical Consultation Support Needed",
        "Medical Treatment Support Needed",
    ],
    "Psychological Support": [
        "Psychological Support Needed After Recent Trauma",
        "Counseling Request for Anxiety and Stress",
        "Support Group Needed for Displaced Person",
        "Mental Health Assistance Request",
        "Psychological Support for Teenagers",
    ],
    "Child & Family Support": [
        "Childcare Support Needed During Relocation",
        "Family Assistance Needed for School Enrollment",
        "Support Needed for Single Parent Household",
        "Help Needed with Baby Essentials",
    ],
    "Legal / Administrative Assistance": [
        "Help Needed with Documents and Registration",
        "Assistance Needed for IDP Status Application",
        "Legal Consultation Needed for Housing Paperwork",
        "Support Needed for Benefit Application",
    ],
    "Employment / Livelihoods": [
        "Job Search Support Needed After Relocation",
        "Help Needed Updating CV and Applying for Jobs",
        "Livelihood Assistance Request for Displaced Worker",
        "Short-Term Work Opportunity Support Needed",
    ],
    "Education / Tutoring": [
        "Tutoring Support Needed for School Child",
        "Online Lessons Needed for Exam Preparation",
        "Homework Help Needed for School Child",
        "English Tutoring Request for Teen Student",
    ],
    "Evacuation / Relocation": [
        "Evacuation Needed for Family with Child",
        "Assistance Needed for Relocation to Safer City",
        "Evacuation Support Needed Within 24 Hours",
        "Relocation Coordination Needed for Family",
        "Help Needed with Moving Belongings",
    ],
    "Volunteer Coordination / Community Support": [
        "Volunteers Needed for Community Supply Distribution",
        "Looking for Volunteers to Support Local Shelter",
        "Coordination Help Needed for Aid Delivery",
        "Community Support Needed for Sorting Donations",
    ],
    "Animal / Pet Assistance": [
        "Temporary Foster Needed for Cat",
        "Pet Food Needed for Displaced Family",
        "Veterinary Support Needed for Injured Dog",
        "Help Needed to Evacuate with Pets",
    ],
    "Other": [
        "General Assistance Request",
        "Help Needed with Urgent Situation",
        "Support Request (Other)",
        "Assistance Needed — Please Contact",
    ],
}

# request-type descriptions (kind = "request")
DESCRIPTION_SNIPPETS = [
    "Assistance is needed as soon as possible.",
    "An elderly woman (72) is looking for temporary housing for one month. No special medical needs.",
    "Adult male requires assistance with medications for chronic heart condition. Regular supply needed.",
    "Limited mobility, minimal luggage.",
    "A family of three with a 4-year-old child needs evacuation from frontline area. Limited belongings. Assistance required urgently.",
    "No special needs reported.",
    "Group online therapy sessions needed for teenagers affected by war-related stress.",
    "Arrival expected within the next few days.",
    "One children (age 2).",
    "Two children (ages 1 and 3).",
    "Two children (ages 5 and 9).",
    "Two children (ages 2 and 3,5).",
    "Two children (ages 0.5 and 5).",
    "Request is time-sensitive due to safety concerns.",
    "We can provide additional details upon contact.",
    "Preferred contact via phone or email.",
    "Family of five displaced from Donetsk region needs food and basic hygiene supplies.",
    "Support needed within the next 48 hours.",
    "Flexible schedule, can coordinate pick-up time.",
]

# Offer-type descriptions (kind = "offer")
OFFER_DESCRIPTIONS = [
    "I can provide support in this category. Please reach out to coordinate details.",
    "Available to help on weekends and evenings. Can travel within the city.",
    "I have experience and resources to assist. Contact me for availability.",
    "Offering help for urgent cases. Response time within a few hours.",
]

## **1.6. Creating a data model class**

In [ ]:
# Data models

@dataclass
class UserRow:
    id: int
    email: str
    phone_number: str
    first_name: str
    last_name: str
    password: str
    role: str  # distressed | volunteer | admin
    date_joined: Optional[str]

@dataclass
class CategoryRow:
    id: int
    name: str

@dataclass
class HelpRow:
    id: int
    title: str
    location: str
    description: str
    kind: str  # request | offer
    category_id: int
    status: str  # new | in_progress | done
    creator_id: int
    counterpart_id: Optional[int]
    created_at: str
    completed_at: Optional[str]

@dataclass
class EventRow:
    event_id: int
    event_name: str
    occurred_at: str
    user_id: Optional[int]            # actor
    user_role: Optional[str]          # actor role
    help_id: Optional[int]
    kind: Optional[str]               # request | offer
    category_id: Optional[int]
    location: Optional[str]
    status: Optional[str]
    old_status: Optional[str]
    new_status: Optional[str]
    counterpart_id: Optional[int]
    results_count: Optional[int]
    entry_point: Optional[str]
    cta_location: Optional[str]
    error_type: Optional[str]
    channel: Optional[str]
    delivery_status: Optional[str]
    failure_reason: Optional[str]

## **1.7. Creating functions to determine user names and surnames, emails and set their corresponding roles**

In [ ]:
# Generator

EMAIL_DOMAINS = ["gmail.com", "ukr.net", "i.ua", "meta.ua", "online.ua", "bigmir.net"]

def generate_users(n_users: int, rng: random.Random) -> pd.DataFrame:
    """
    Creating a user pool with only one administrator.
    The rest is distributed among users and volunteers.
    """
    if n_users < 1:
        raise ValueError("n_users must be >= 1")

    first_names = ["Sophia", "Olena", "Iryna", "Kateryna", "Andrii", "Dmytro", "Serhii",
                   "Maksym", "Yulia", "Oksana", "Ivan", "Petro", "Glib", "Anastasia",
                   "Daria", "Vadim"]
    last_names = ["Shevchenko", "Koval", "Bondarenko", "Tkachenko", "Kravchenko",
                  "Polishchuk", "Boyko", "Melnyk", "Ivanchenko", "Klimenko",
                  "Makarov", "Sviridenko", "Petrenko"]

    def make_email(fn: str, ln: str, user_id: int) -> str:
        domain = rng.choice(EMAIL_DOMAINS)
        local_part = f"{slugify(fn)}.{slugify(ln)}{user_id}"
        return f"{local_part}@{domain}"

    rows: List[UserRow] = []

    now = datetime.utcnow()
    users_start = now - timedelta(days=180)

    # admin
    fn = rng.choice(first_names)
    ln = rng.choice(last_names)
    email = make_email(fn, ln, 1)
    phone = generate_ua_phone(rng)
    date_joined = rand_dt_between(users_start, now, rng)

    rows.append(
        UserRow(
            id=1,
            email=email,
            phone_number=phone,
            first_name=fn,
            last_name=ln,
            password="hashed_password_placeholder",
            role="admin",
            date_joined=date_joined.isoformat(),
        )
    )

    # distressed & volunteers
    roles = [("distressed", 0.70), ("volunteer", 0.30)]

    for i in range(2, n_users + 1):
        fn = rng.choice(first_names)
        ln = rng.choice(last_names)
        role = pick_weighted(roles, rng)
        email = make_email(fn, ln, i)
        phone = generate_ua_phone(rng)
        date_joined = rand_dt_between(users_start, now, rng)

        rows.append(
            UserRow(
                id=int(i),
                email=email,
                phone_number=phone,
                first_name=fn,
                last_name=ln,
                password="hashed_password_placeholder",
                role=role,
                date_joined=date_joined.isoformat(),
            )
        )

    return pd.DataFrame([r.__dict__ for r in rows])

## **1.8. Creating functions to determine categories and set their title and description**

In [ ]:
# Generator
def generate_categories(categories: List[str]) -> pd.DataFrame:
    rows = [CategoryRow(id=int(i + 1), name=cat) for i, cat in enumerate(categories)]
    return pd.DataFrame([r.__dict__ for r in rows])


def build_title_and_description(category_name: str, city: str, kind: str, rng: random.Random) -> Tuple[str, str]:
    title_options = TITLE_TEMPLATES.get(category_name, TITLE_TEMPLATES["Other"])
    title = rng.choice(title_options)

    if kind == "offer":
        # Offers
        desc = rng.choice(OFFER_DESCRIPTIONS)
        desc += f" Location: {city}."
        return title, desc

    # Requests
    core_lines = [
        f"Request in {city}.",
        rng.choice(DESCRIPTION_SNIPPETS),
        rng.choice(DESCRIPTION_SNIPPETS),
    ]
    # Make it a bit more realistic and readable
    description = " ".join(core_lines)
    return title, description

## **1.9. Creating functions to determine help and events for analytics**

In [ ]:
def generate_help_and_events(
    users_df: pd.DataFrame,
    categories_df: pd.DataFrame,
    n_help: int,
    rng: random.Random,
    generate_events: bool = True,
    add_form_events: bool = True,
    abandoned_form_rate: float = 0.12,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Generator:
  - Ensures that the help table logic is consistent:
    if request_declined => final help.status='new' AND help.counterpart_id=NULL.
  - Adds request/offer form funnel events:
    request_form_started / request_form_submitted.
    """

    requester_ids = users_df.loc[users_df["role"] == "distressed", "id"].astype(int).tolist()
    volunteer_ids = users_df.loc[users_df["role"] == "volunteer", "id"].astype(int).tolist()
    if not requester_ids:
        raise ValueError("Need at least 1 distressed with role='distressed'")
    if not volunteer_ids:
        raise ValueError("Need at least 1 distressed with role='volunteer'")

    cat_ids = categories_df["id"].astype(int).tolist()
    cat_id_to_name = dict(zip(categories_df["id"].astype(int), categories_df["name"].astype(str)))

    now = utc_now()
    start = now - timedelta(days=90)

    help_rows: List[HelpRow] = []
    event_rows: List[dict] = []
    event_id = 1

    def add_event(
        event_name: str,
        occurred_at: datetime,
        user_id: Optional[int],
        user_role: Optional[str],
        help_id: Optional[int],
        kind: Optional[str],
        category_id: Optional[int],
        location: Optional[str],
        status: Optional[str] = None,
        old_status: Optional[str] = None,
        new_status: Optional[str] = None,
        counterpart_id: Optional[int] = None,
        results_count: Optional[int] = None,
        entry_point: Optional[str] = None,
        cta_location: Optional[str] = None,
        error_type: Optional[str] = None,
        channel: Optional[str] = None,
        delivery_status: Optional[str] = None,
        failure_reason: Optional[str] = None,
    ):
        nonlocal event_id
        if not generate_events:
            return

        event_rows.append(
            EventRow(
                event_id=int(event_id),
                event_name=event_name,
                occurred_at=occurred_at.isoformat(),
                user_id=int(user_id) if user_id is not None else None,
                user_role=user_role,
                help_id=int(help_id) if help_id is not None else None,
                kind=kind,
                category_id=int(category_id) if category_id is not None else None,
                location=location,
                status=status,
                old_status=old_status,
                new_status=new_status,
                counterpart_id=int(counterpart_id) if counterpart_id is not None else None,
                results_count=int(results_count) if results_count is not None else None,
                entry_point=entry_point,
                cta_location=cta_location,
                error_type=error_type,
                channel=channel,
                delivery_status=delivery_status,
                failure_reason=failure_reason,
            ).__dict__
        )
        event_id += 1

    # 1) Abandoned request forms (no help_id)
    if add_form_events and abandoned_form_rate > 0:
        n_abandoned = int(round(n_help * abandoned_form_rate))
        for _ in range(n_abandoned):
            uid = int(rng.choice(requester_ids))
            category_id = int(rng.choice(cat_ids))
            city = rng.choice(CITIES)
            t0 = rand_dt_between(start, now - timedelta(hours=1), rng)
            t1 = t0 + timedelta(seconds=rng.randint(10, 600))

            add_event(
                event_name="request_form_started",
                occurred_at=t0,
                user_id=uid,
                user_role="distressed",
                help_id=None,
                kind="request",
                category_id=category_id,
                location=city,
                entry_point="home",
            )
            add_event(
                event_name="request_form_submitted",
                occurred_at=t1,
                user_id=uid,
                user_role="distressed",
                help_id=None,
                kind="request",
                category_id=category_id,
                location=city,
                entry_point="home",
                error_type="validation_error" if rng.random() < 0.4 else "abandoned",
            )

    # 2) Main help generation
    kind_weights = [("request", 0.75), ("offer", 0.25)]

    for hid in range(1, n_help + 1):
        kind = pick_weighted(kind_weights, rng)
        category_id = int(rng.choice(cat_ids))
        category_name = cat_id_to_name[category_id]
        city = rng.choice(CITIES)

        creator_id = int(rng.choice(requester_ids if kind == "request" else volunteer_ids))

        created_at = rand_dt_between(start, now - timedelta(hours=1), rng)

        # 2a) Form funnel events
        if add_form_events:
            form_started = created_at - timedelta(seconds=rng.randint(20, 900))
            form_submitted = form_started + timedelta(seconds=rng.randint(10, 600))

            if kind == "request":
                add_event(
                    "request_form_started", form_started, creator_id, "distressed",
                    None, "request", category_id, city, entry_point="home"
                )
                add_event(
                    "request_form_submitted", form_submitted, creator_id, "distressed",
                    None, "request", category_id, city, entry_point="home"
                )
            else:
                add_event(
                    "offer_form_started", form_started, creator_id, "volunteer",
                    None, "offer", category_id, city, entry_point="profile"
                )
                add_event(
                    "offer_form_submitted", form_submitted, creator_id, "volunteer",
                    None, "offer", category_id, city, entry_point="profile"
                )

        title, description = build_title_and_description(category_name, city, kind, rng)

        status = "new"
        counterpart_id: Optional[int] = None
        completed_at: Optional[datetime] = None

        if kind == "request":
            # lifecycle decisions
            taken = rng.random() < 0.45
            completed = taken and (rng.random() < 0.55)
            declined = taken and (not completed) and (rng.random() < 0.20)

            # request created
            add_event(
                "request_created",
                created_at, creator_id, "distressed",
                hid, "request", category_id, city,
                status="new",
            )

            if taken:
                counterpart_id = int(rng.choice(volunteer_ids))
                take_time = rand_dt_between(
                    created_at + timedelta(minutes=1),
                    min(created_at + timedelta(days=3), now),
                    rng,
                )

                add_event(
                    "request_take_attempted",
                    take_time - timedelta(seconds=rng.randint(5, 120)),
                    counterpart_id, "volunteer",
                    hid, "request", category_id, city,
                    status="new",
                    counterpart_id=counterpart_id,
                    entry_point="help_list",
                    cta_location="help_card",
                )
                add_event(
                    "request_take_succeeded",
                    take_time,
                    counterpart_id, "volunteer",
                    hid, "request", category_id, city,
                    status="in_progress",
                    counterpart_id=counterpart_id,
                    entry_point="help_details",
                    cta_location="take_button",
                )
                add_event(
                    "request_status_changed",
                    take_time,
                    counterpart_id, "volunteer",
                    hid, "request", category_id, city,
                    old_status="new",
                    new_status="in_progress",
                    counterpart_id=counterpart_id,
                )

                if completed:
                    completed_at = rand_dt_between(take_time + timedelta(minutes=10), now, rng)
                    status = "done"

                    add_event(
                        "request_completed",
                        completed_at,
                        counterpart_id, "volunteer",
                        hid, "request", category_id, city,
                        status="done",
                        counterpart_id=counterpart_id,
                    )
                    add_event(
                        "request_status_changed",
                        completed_at,
                        counterpart_id, "volunteer",
                        hid, "request", category_id, city,
                        old_status="in_progress",
                        new_status="done",
                        counterpart_id=counterpart_id,
                    )

                elif declined:
                    decline_time = rand_dt_between(take_time + timedelta(minutes=5), now, rng)

                    # cleared in the reference table
                    status = "new"
                    counterpart_id = None

                    add_event(
                        "request_declined",
                        decline_time,
                        int(rng.choice(volunteer_ids)), "volunteer",
                        hid, "request", category_id, city,
                        status="new",
                        counterpart_id=None,
                    )
                    add_event(
                        "request_status_changed",
                        decline_time,
                        int(rng.choice(volunteer_ids)), "volunteer",
                        hid, "request", category_id, city,
                        old_status="in_progress",
                        new_status="new",
                        counterpart_id=None,
                    )
                else:
                    status = "in_progress"

        else:
            # Offer
            add_event(
                "offer_created",
                created_at, creator_id, "volunteer",
                hid, "offer", category_id, city,
                status="new",
            )

            matched = rng.random() < 0.15
            if matched:
                counterpart_id = int(rng.choice(requester_ids))
                status = "done"
                completed_at = rand_dt_between(created_at + timedelta(hours=1), now, rng)

                # "offer_completed": "offer_matched"
                add_event(
                    "offer_matched",
                    completed_at, creator_id, "volunteer",
                    hid, "offer", category_id, city,
                    status="done",
                    counterpart_id=counterpart_id,
                )

        help_rows.append(
            HelpRow(
                id=int(hid),
                title=title,
                location=city,
                description=description,
                kind=kind,
                category_id=int(category_id),
                status=status,
                creator_id=int(creator_id),
                counterpart_id=int(counterpart_id) if counterpart_id is not None else None,
                created_at=created_at.isoformat(),
                completed_at=completed_at.isoformat() if completed_at is not None else None,
            )
        )

    help_df = pd.DataFrame([r.__dict__ for r in help_rows])
    help_df["id"] = help_df["id"].astype("int64")
    help_df["category_id"] = help_df["category_id"].astype("int64")
    help_df["creator_id"] = help_df["creator_id"].astype("int64")
    help_df["counterpart_id"] = help_df["counterpart_id"].astype("Int64")

    events_df = pd.DataFrame(event_rows) if generate_events else pd.DataFrame()
    if not events_df.empty:
        for col in ["event_id", "user_id", "help_id", "category_id", "counterpart_id", "results_count"]:
            if col in events_df.columns:
                events_df[col] = events_df[col].astype("Int64")
        events_df["event_id"] = events_df["event_id"].astype("int64")

    return help_df, events_df


# Validation helpers
def validate_dataset(help_df: pd.DataFrame, events_df: pd.DataFrame) -> list[tuple]:
    issues = []

    # 1) 'new' and counterpart_id NULL
    declined = events_df[events_df["event_name"] == "request_declined"]
    if not declined.empty:
        ids = declined["help_id"].dropna().astype(int).unique()
        sub = help_df[help_df["id"].isin(ids)]
        bad = sub[(sub["kind"] == "request") & ((sub["status"] != "new") | (~sub["counterpart_id"].isna()))]
        if len(bad) > 0:
            issues.append(("decline_final_state", len(bad), bad.head(5)))

    # 2) done
    done = help_df[help_df["status"] == "done"]
    bad = done[done["completed_at"].isna()]
    if len(bad) > 0:
        issues.append(("done_missing_completed_at", len(bad), bad.head(5)))

    # 3) form funnel events exist
    if events_df[events_df["event_name"] == "request_form_started"].empty:
        issues.append(("missing_request_form_started", None, None))

    return issues

## **1.10. Save dataset "wings_of_help_dataset"**

In [ ]:
def main(
    output_dir: str = "wings_of_help_dataset",
    n_users: int = 500,
    n_help: int = 2000,
    seed: int = 42,
    with_events: bool = True,
) -> None:
    rng = random.Random(seed)

    categories_df = generate_categories(CATEGORIES)
    users_df = generate_users(n_users, rng)

    help_df, events_df = generate_help_and_events(
        users_df=users_df,
        categories_df=categories_df,
        n_help=n_help,
        rng=rng,
        generate_events=with_events,
        add_form_events=True,
        abandoned_form_rate=0.12,
    )

    os.makedirs(output_dir, exist_ok=True)
    users_df.to_csv(os.path.join(output_dir, "users.csv"), index=False, encoding="utf-8")
    categories_df.to_csv(os.path.join(output_dir, "help_categories.csv"), index=False, encoding="utf-8")
    help_df.to_csv(os.path.join(output_dir, "help.csv"), index=False, encoding="utf-8")
    if with_events:
        events_df.to_csv(os.path.join(output_dir, "events.csv"), index=False, encoding="utf-8")

    # quick check
    if with_events:
        issues = validate_dataset(help_df, events_df)
        if issues:
            print("VALIDATION ISSUES:")
            for name, count, sample in issues:
                print("-", name, count)
        else:
            print("Validation OK")

    print(f"Saved dataset to: {output_dir}")
    print(f"- help_categories.csv: {len(categories_df):,} rows")
    print(f"- users.csv:          {len(users_df):,} rows")
    print(f"- help.csv:           {len(help_df):,} rows")

    if with_events:
        print(f"- events.csv:         {len(events_df):,} rows")

if __name__ == "__main__":
    main()

/tmp/ipython-input-799642207.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


Validation OK
Saved dataset to: wings_of_help_dataset
- help_categories.csv: 13 rows
- users.csv:          500 rows
- help.csv:           2,000 rows
- events.csv:         9,510 rows


## **1.11. Download "wings_of_help_dataset"**

In [ ]:
from google.colab import files

files.download("wings_of_help_dataset/events.csv")
files.download("wings_of_help_dataset/help_categories.csv")
files.download("wings_of_help_dataset/users.csv")
files.download("wings_of_help_dataset/help.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>